GPT: Generative Pre-training Transformer Model
====

In [1]:
from itertools import islice, zip_longest
import math
from pathlib import Path
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split
from tokenizers import Tokenizer
from tqdm.auto import tqdm

## Model components


GPT uses a stack of decoder-style Transformer blocks. Each block contains masked multi-head self-attention and a position-wise feed-forward network, but no encoder-decoder cross-attention.

- 12-layer transformer decoder
- 768 embedding dim
- 12 attention heads
- FFN 3072 dim
- GELU activation
- Weight initialization N(0, 0.02)
- Learned position embeddings (instead of sinusoidal)


### Implementation notes:
* In the transformer implementation, we put the residual dropout in the encoder/decoder blocks. But here, keeping it inside attention mirrors the reference implementation more closely. GPT-1 uses 0.1 for attention, residual, and embedding dropout.

* Edge case for padding mask: using `-inf` can produce `NaN` when a query has every available key masked, which can happen in the early positions of a left-padded sequence. It is safe if you consistently use right padding with at least one valid token. A safer option is to use
```python
  mask_value = torch.finfo(attn_scores.dtype).min
```



In [2]:

class MultiheadAttention(nn.Module):
    def __init__(
        self,
        embedding_dim,
        num_heads,
        context_length,
        dropout_rate=0.1,
    ):
        super().__init__()
        if embedding_dim % num_heads != 0:
            raise ValueError(
                f"embedding_dim ({embedding_dim}) must be divisible by "
                f"num_heads ({num_heads})"
            )

        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads

        self.w_attn = nn.Linear(embedding_dim, 3 * embedding_dim)  # for q, k & v
        # self.w_q = nn.Linear(embedding_dim, embedding_dim)
        # self.w_k = nn.Linear(embedding_dim, embedding_dim)
        # self.w_v = nn.Linear(embedding_dim, embedding_dim)
        self.w_o = nn.Linear(embedding_dim, embedding_dim)
        self.attn_dropout = nn.Dropout(dropout_rate)
        self.resid_dropout = nn.Dropout(dropout_rate)

        causal_mask = torch.triu(
            torch.ones(context_length, context_length, dtype=torch.bool),
            diagonal=1,
        )
        self.register_buffer("causal_mask", causal_mask, persistent=False)

    def forward(self, hidden_states, padding_mask=None):
        batch_size, seq_len, embedding_dim = hidden_states.shape
        assert embedding_dim == self.embedding_dim

        q, k, v = self.w_attn(hidden_states).split(self.embedding_dim, dim=-1)
        # q = self.w_q(hidden_states)
        # k = self.w_k(hidden_states)
        # v = self.w_v(hidden_states)

        q = q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = q @ k.transpose(-2, -1) / (self.head_dim**0.5)

        assert seq_len <= self.causal_mask.shape[0]
        causal_mask = self.causal_mask[:seq_len, :seq_len]
        attn_scores = attn_scores.masked_fill(
            causal_mask[None, None, :, :], float("-inf")
        )

        if padding_mask is not None:
            assert padding_mask.shape == (batch_size, seq_len)
            assert padding_mask.dtype == torch.bool
            mask_value = torch.finfo(attn_scores.dtype).min
            attn_scores = attn_scores.masked_fill(
                padding_mask[:, None, None, :], mask_value
            )

        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)
        output = attn_weights @ v
        output = output.transpose(1, 2).contiguous().view(
            batch_size, seq_len, embedding_dim
        )
        output = self.w_o(output)
        return self.resid_dropout(output)

* Assigning weights
The transpose is necessary because Hugging Face’s legacy `Conv1D` stores weights as `[input_dim, output_dim]`, while `nn.Linear` stores them as `[output_dim, input_dim]`.


* Padding mask: 
Our implementation expects:
```
True  = padding/blocked
False = valid token
```
Hugging Face attention masks use the opposite public convention: 1 means valid and 0 means masked. A future weight-loading or parity cell should convert between them explicitly.

### GPT Block

* GPT replaces ReLU activation with GELU activation function
* Still using post-layernorm as in the original transformer

In [3]:
class GPTBlock(nn.Module):
    def __init__(
            self,
            *,
            embedding_dim,
            num_heads,
            context_length,
            dropout_rate=0.1
        ):
        super().__init__()

        self.causal_self_attention = MultiheadAttention(
            embedding_dim,
            num_heads,
            context_length,
            dropout_rate=dropout_rate,
        )
        self.feed_forward = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.GELU(approximate="tanh"),
            nn.Linear(4 * embedding_dim, embedding_dim),
            nn.Dropout(dropout_rate)
        )
        self.attn_layernorm = nn.LayerNorm(embedding_dim, eps=1e-5)
        self.ffn_layernorm = nn.LayerNorm(embedding_dim, eps=1e-5)

    def forward(
            self,
            hidden_states,
            padding_mask=None,
        ):
        attn_output = self.causal_self_attention(
            hidden_states, padding_mask=padding_mask
        )
        hidden_states = self.attn_layernorm(hidden_states + attn_output)

        ffn_output = self.feed_forward(hidden_states)
        return self.ffn_layernorm(hidden_states + ffn_output)

### Embeddings

  `input_ids [batch, sequence]`  
   → token embeddings [batch, sequence, embedding_dim]

  `position_ids [sequence]`  
   → position embeddings [sequence, embedding_dim]

  token embeddings + position embeddings  
   → embedding dropout
   → hidden_states [batch, sequence, embedding_dim]


  Note, we remove `padding_idx` in token embedding to match the published GPT checkpoint.

In [4]:
class GPTEmbeddings(nn.Module):
    def __init__(
            self,
            *,
            vocab_size,
            embedding_dim,
            context_len,
            dropout_rate
        ):
        super().__init__()

        self.token_embeddings = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
        )
        self.position_embeddings = nn.Embedding(
            num_embeddings=context_len,
            embedding_dim=embedding_dim,
        )
        self.embedding_dropout = nn.Dropout(dropout_rate)

    def forward(
            self,
            token_ids: torch.Tensor
        ):
        seq_len = token_ids.size(1)
        tok_embeds = self.token_embeddings(token_ids)
        positions = torch.arange(seq_len, device=token_ids.device)
        pos_embeds = self.position_embeddings(positions)

        return self.embedding_dropout(tok_embeds + pos_embeds)

### GPT Model Full Architecture

 * Base model `GPTModel`
 * Causal-LM wrapper: `GPTForCausalLM`